In [ ]:
import pandas as pd
import re
from bs4 import BeautifulSoup
from typing import List
import os




In [ ]:
import os
import pandas as pd
from bs4 import BeautifulSoup
import re
import warnings


RAW_PATH = "data/raw"
PROCESSED_PATH = "data/processed"
os.makedirs(RAW_PATH, exist_ok=True)
os.makedirs(PROCESSED_PATH, exist_ok=True)


# Ignorer les warnings BeautifulSoup

from bs4 import MarkupResemblesLocatorWarning
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)


# Fonctions utilitaires

def clean_text(text):
    """Nettoie un texte : HTML, URLs, caractères spéciaux, espaces multiples, mise en minuscules"""
    text = BeautifulSoup(str(text), "html.parser").get_text()  # supprimer HTML
    text = re.sub(r"http\S+|www\S+", "", text)               # supprimer URLs
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)             # supprimer caractères spéciaux
    text = re.sub(r"\s+", " ", text)                         # supprimer espaces multiples
    return text.strip().lower()

def chunk_text(text, chunk_size=200, overlap=50):
    """Découpe un texte en chunks avec chevauchement"""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        if chunk:  
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


# Charger et fusionner CSV

csv_files = [f for f in os.listdir(RAW_PATH) if f.endswith(".csv")]
dfs = []

for file_name in csv_files:
    file_path = os.path.join(RAW_PATH, file_name)
    print(f"\nChargement de {file_name}")
    df_tmp = pd.read_csv(file_path)
    
    # Ajouter le label à partir du nom de fichier
    if "fake" in file_name.lower():
        df_tmp['label'] = 'Fake'
    else:
        df_tmp['label'] = 'True'
    
    dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)
print(f"\nTaille totale du dataset : {df.shape}")


# Nettoyage des textes

df['clean_text'] = df['text'].apply(clean_text)


# Découpage en chunks

all_chunks = []
for idx, row in df.iterrows():
    chunks = chunk_text(row['clean_text'], chunk_size=200, overlap=50)
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            'title': row['title'],
            'chunk_id': i,
            'chunk_text': chunk,
            'label': row['label'],
            'date': row.get('date', '')
        })

df_chunks = pd.DataFrame(all_chunks)
print(f"Nombre total de chunks : {df_chunks.shape[0]}")


# Sauvegarde

chunks_file = os.path.join(PROCESSED_PATH, "chunks_dataset.csv")
df_chunks.to_csv(chunks_file, index=False)
print(f"Chunks sauvegardés dans : {chunks_file}")



Chargement de Fake.csv

Chargement de True.csv

Taille totale du dataset : (44898, 5)
Nombre total de chunks : 146255
Chunks sauvegardés dans : data/processed/chunks_dataset.csv


In [9]:
# Exemple : prendre les 5 premiers textes nettoyés
print("Avant mise en minuscules :\n")
print(df['clean_text'].head())

# Appliquer la mise en minuscules
df['clean_text'] = df['clean_text'].str.lower()

print("\nAprès mise en minuscules :\n")
print(df['clean_text'].head())


Avant mise en minuscules :

0    donald trump just couldn t wish all americans ...
1    house intelligence committee chairman devin nu...
2    on friday it was revealed that former milwauke...
3    on christmas day donald trump announced that h...
4    pope francis used his annual christmas day mes...
Name: clean_text, dtype: object

Après mise en minuscules :

0    donald trump just couldn t wish all americans ...
1    house intelligence committee chairman devin nu...
2    on friday it was revealed that former milwauke...
3    on christmas day donald trump announced that h...
4    pope francis used his annual christmas day mes...
Name: clean_text, dtype: object


# Supprimer les balises HTML

In [ ]:
df = pd.read_csv("/home/marwa/Fake-News-Detect/data/Fake.csv")  
df = pd.read_csv("/home/marwa/Fake-News-Detect/data/True.csv")  


print(df)
df.head()
def remove_html_tags(text):
    """Supprime les balises HTML avec BeautifulSoup"""
    return BeautifulSoup(text, "html.parser").get_text()


                                                   title  \
0      As U.S. budget fight looms, Republicans flip t...   
1      U.S. military to accept transgender recruits o...   
2      Senior U.S. Republican senator: 'Let Mr. Muell...   
3      FBI Russia probe helped by Australian diplomat...   
4      Trump wants Postal Service to charge 'much mor...   
...                                                  ...   
21412  'Fully committed' NATO backs new U.S. approach...   
21413  LexisNexis withdrew two products from Chinese ...   
21414  Minsk cultural hub becomes haven from authorities   
21415  Vatican upbeat on possibility of Pope Francis ...   
21416  Indonesia to buy $1.14 billion worth of Russia...   

                                                    text       subject  \
0      WASHINGTON (Reuters) - The head of a conservat...  politicsNews   
1      WASHINGTON (Reuters) - Transgender people will...  politicsNews   
2      WASHINGTON (Reuters) - The special counsel inv... 

In [ ]:


df = pd.read_csv("/home/marwa/Fake-News-Detect/data/Fake.csv")  
df_True = pd.read_csv("/home/marwa/Fake-News-Detect/data/True.csv")  



print(df.head())
print(df.columns)



print(df_True .head())
print(df_True .columns)


                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date  
0  December 31, 2017  
1  December 31, 2017  
2  December 30, 2017  
3  December 29, 2017  
4  December 25, 2017  
Index(['title', 'text', 'subject', 'date'], dtype='object')
                                               title  \
0  As U.S. budget fight 

In [ ]:
import re
from bs4 import BeautifulSoup
from bs4 import MarkupResemblesLocatorWarning
import warnings

# Ignorer les warnings liés aux URLs
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Supprimer les balises HTML
    text = BeautifulSoup(text, "html.parser").get_text()
    
    # 2. Supprimer les URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # 3. Supprimer les caractères spéciaux (laisser lettres, chiffres et espaces)
    text = re.sub(r'[^A-Za-z0-9À-ÿ\s]', '', text)
    
    # 4. Supprimer les espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 5. Conversion en minuscules
    text = text.lower()
    
    return text

# Appliquer le nettoyage sur la colonne 'text'
df['clean_text'] = df['text'].apply(clean_text)

# Vérifier le résultat
print(df[['text', 'clean_text']].head(5))


                                                text  \
0  Donald Trump just couldn t wish all Americans ...   
1  House Intelligence Committee Chairman Devin Nu...   
2  On Friday, it was revealed that former Milwauk...   
3  On Christmas day, Donald Trump announced that ...   
4  Pope Francis used his annual Christmas Day mes...   

                                          clean_text  
0  donald trump just couldn t wish all americans ...  
1  house intelligence committee chairman devin nu...  
2  on friday it was revealed that former milwauke...  
3  on christmas day donald trump announced that h...  
4  pope francis used his annual christmas day mes...  
